# Verificación entorno

In [8]:
import sys
import os
import subprocess
import shutil
from pathlib import Path

print(f"Python del kernel: {sys.version}")
print()

# Entorno limpio: elimina el backend de Jupyter que rompe matplotlib
# en subprocesses donde matplotlib_inline no está instalado
clean_env = os.environ.copy()
clean_env["MPLBACKEND"] = "Agg"

print("✅ clean_env configurado (MPLBACKEND=Agg)")

Python del kernel: 3.14.4 (main, Apr  7 2026, 13:13:20) [Clang 21.0.0 (clang-2100.0.123.102)]

✅ clean_env configurado (MPLBACKEND=Agg)


# Instalación python 3.12

In [9]:
brew_check = subprocess.run(
    ["which", "brew"], capture_output=True, text=True, env=clean_env
)

if brew_check.returncode != 0:
    print("❌ Homebrew no encontrado — instalar desde https://brew.sh")
else:
    print(f"✅ Homebrew: {brew_check.stdout.strip()}")
    print("Instalando python@3.12 (puede tardar 2-5 min)...")

    result = subprocess.run(
        ["brew", "install", "python@3.12"],
        text=True,
        env=clean_env
    )

    if result.returncode == 0:
        print("\n✅ python@3.12 instalado")
    else:
        print("\n⚠️  Puede que ya estuviera instalado — continuar igual")

✅ Homebrew: /opt/homebrew/bin/brew
Instalando python@3.12 (puede tardar 2-5 min)...


✔︎ JSON API packages.arm64_tahoe.jws.json



✅ python@3.12 instalado


To reinstall 3.12.13_4, run:
  brew reinstall python@3.12


# Detectar python 3.12 y crear venv 

In [10]:
CANDIDATES = [
    "/opt/homebrew/bin/python3.12",
    "/usr/local/bin/python3.12",
    "/opt/homebrew/opt/python@3.12/bin/python3.12",
    shutil.which("python3.12"),
]

python312 = next((p for p in CANDIDATES if p and Path(p).exists()), None)

if not python312:
    print("❌ Python 3.12 no encontrado.")
    print("Ejecutar en Terminal: brew install python@3.12")
    print("Luego reiniciar el kernel y volver a correr desde Celda 1.")
else:
    ver = subprocess.run(
        [python312, "--version"], capture_output=True, text=True, env=clean_env
    )
    print(f"✅ Python 3.12: {python312}")
    print(f"   Versión: {ver.stdout.strip()}")

    EXPORT_VENV    = Path("venv_export312")
    pip312         = EXPORT_VENV / "bin" / "pip"
    python312_venv = EXPORT_VENV / "bin" / "python"

    if not EXPORT_VENV.exists():
        print(f"\nCreando venv en {EXPORT_VENV}...")
        result = subprocess.run(
            [python312, "-m", "venv", str(EXPORT_VENV)],
            capture_output=True, text=True, env=clean_env
        )
        if result.returncode == 0:
            print("✅ venv creado")
        else:
            print(f"❌ Error:\n{result.stderr}")
    else:
        print(f"\n✅ venv ya existe en {EXPORT_VENV}")

    print(f"\nPython export: {python312_venv}")

✅ Python 3.12: /opt/homebrew/bin/python3.12
   Versión: Python 3.12.13

✅ venv ya existe en venv_export312

Python export: venv_export312/bin/python


# Instalar dependencias

In [11]:
print("Instalando ultralytics + tensorflow (puede tardar 3-5 min)...\n")

install = subprocess.run(
    [str(pip312), "install", "--quiet",
     "ultralytics", "tensorflow", "onnx", "onnxslim"],
    capture_output=True, text=True, env=clean_env
)

if install.returncode == 0:
    print("✅ Dependencias instaladas")
else:
    print(f"⚠️  pip output:\n{install.stderr[-1000:]}")

# Verificar TensorFlow con entorno limpio
check = subprocess.run(
    [str(python312_venv), "-c",
     "import tensorflow as tf; print(tf.__version__)"],
    capture_output=True, text=True, env=clean_env
)

if check.returncode == 0:
    print(f"✅ TensorFlow {check.stdout.strip()} listo en venv_export312")
else:
    print(f"❌ TensorFlow falla:\n{check.stderr[-800:]}")

Instalando ultralytics + tensorflow (puede tardar 3-5 min)...

✅ Dependencias instaladas
✅ TensorFlow 2.19.1 listo en venv_export312


# Exportar YOLOv8n a TFLite INT8 @ 96×96

In [8]:
EXPORT_SCRIPT = """
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
result = model.export(
    format="tflite",
    imgsz=96,
    int8=True,
    data="coco128.yaml",
)
print(f"__RESULT__:{result}")
"""

print("Exportando YOLOv8n → TFLite INT8 @ 96×96...")
print("(~1-2 minutos)\n")

result = subprocess.run(
    [str(python312_venv), "-c", EXPORT_SCRIPT],
    capture_output=True, text=True,
    cwd=Path(".").resolve(),
    env=clean_env
)

print(result.stdout[-3000:] if result.stdout else "(sin output)")

if result.returncode != 0:
    print(f"\n❌ Error:\n{result.stderr[-2000:]}")
else:
    print("\n✅ Export completado")

Exportando YOLOv8n → TFLite INT8 @ 96×96...
(~1-2 minutos)

 dtype=tf.float32, name=None)
  5280602384: TensorSpec(shape=(1, 1, 64, 64), dtype=tf.float32, name=None)
  5280597584: TensorSpec(shape=(1, 1, 64, 64), dtype=tf.float32, name=None)
  5278314192: TensorSpec(shape=(1, 1, 64, 64), dtype=tf.float32, name=None)
  5280602768: TensorSpec(shape=(64,), dtype=tf.float32, name=None)
  5280597008: TensorSpec(shape=(64,), dtype=tf.float32, name=None)
  5278313616: TensorSpec(shape=(64,), dtype=tf.float32, name=None)
  5280601616: TensorSpec(shape=(3, 3, 256, 80), dtype=tf.float32, name=None)
  5278316688: TensorSpec(shape=(3, 3, 128, 80), dtype=tf.float32, name=None)
  5278312656: TensorSpec(shape=(3, 3, 64, 80), dtype=tf.float32, name=None)
  5280601040: TensorSpec(shape=(80,), dtype=tf.float32, name=None)
  5278318416: TensorSpec(shape=(80,), dtype=tf.float32, name=None)
  5278310736: TensorSpec(shape=(80,), dtype=tf.float32, name=None)
  5280602192: TensorSpec(shape=(3, 3, 80, 80), dty

# Localizar el .tflite y verificar tamaño

In [9]:
search_dirs = [Path("yolov8n_saved_model"), Path(".")]

tflite_path = None
for d in search_dirs:
    if d.exists():
        candidates = list(d.rglob("*.tflite"))
        for c in candidates:
            if "int8" in c.name or "full_integer" in c.name:
                tflite_path = c
                break
        if not tflite_path and candidates:
            tflite_path = candidates[0]
    if tflite_path:
        break

if not tflite_path or not tflite_path.exists():
    print("❌ .tflite no encontrado — revisar output de Celda 5")
else:
    size_kb = tflite_path.stat().st_size / 1024
    arena_kb = size_kb * 2.5
    total_kb = size_kb + arena_kb

    print(f"✅ Modelo: {tflite_path}")
    print(f"   Tamaño: {size_kb:.1f} KB ({size_kb/1024:.2f} MB)")
    print()
    print("   Estimación PSRAM (ESP32S3 = 8192 KB):")
    print(f"   Modelo (flash):       {size_kb:>7.0f} KB")
    print(f"   Arena activaciones:   {arena_kb:>7.0f} KB  (×2.5)")
    print(f"   Total estimado:       {total_kb:>7.0f} KB")
    print(f"   Margen libre:         {8192 - total_kb:>7.0f} KB")
    print()

    if total_kb < 8192 * 0.85:
        print("   ✅ DEBERÍA CABER → continuar con firmware Arduino")
    else:
        print("   ⚠️  MARGEN JUSTO → verificar en hardware")
        print("       Si AllocateTensors falla → activar Plan B")

✅ Modelo: yolov8n_saved_model/yolov8n_full_integer_quant.tflite
   Tamaño: 3303.8 KB (3.23 MB)

   Estimación PSRAM (ESP32S3 = 8192 KB):
   Modelo (flash):          3304 KB
   Arena activaciones:      8260 KB  (×2.5)
   Total estimado:         11563 KB
   Margen libre:           -3371 KB

   ⚠️  MARGEN JUSTO → verificar en hardware
       Si AllocateTensors falla → activar Plan B


# Extraer parámetros de cuantización


In [11]:
import json

QUANT_SCRIPT = f"""
import tensorflow as tf, json

interpreter = tf.lite.Interpreter(model_path=r"{tflite_path}")
interpreter.allocate_tensors()

inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

print(json.dumps({{
    "input_shape":       [int(x) for x in inp["shape"]],
    "input_scale":       float(inp["quantization"][0]),
    "input_zero_point":  int(inp["quantization"][1]),
    "output_shape":      [int(x) for x in out["shape"]],
    "output_scale":      float(out["quantization"][0]),
    "output_zero_point": int(out["quantization"][1]),
}}))
"""

quant_result = subprocess.run(
    [str(python312_venv), "-c", QUANT_SCRIPT],
    capture_output=True, text=True, env=clean_env
)

if quant_result.returncode != 0:
    print(f"❌ Error:\n{quant_result.stderr[-800:]}")
else:
    # Saltar líneas previas al JSON (ej: "mps" que imprime TF/Ultralytics)
    json_line = next(
        (l for l in quant_result.stdout.strip().splitlines() if l.startswith("{")),
        None
    )
    if not json_line:
        print("❌ No se encontró JSON en el output:")
        print(quant_result.stdout)
    else:
        info = json.loads(json_line)

        print("Parámetros del modelo:")
        print(f"  Input  shape     : {info['input_shape']}")
        print(f"  Input  scale/zp  : {info['input_scale']:.6f} / {info['input_zero_point']}")
        print(f"  Output shape     : {info['output_shape']}")
        print(f"  Output scale/zp  : {info['output_scale']:.6f} / {info['output_zero_point']}")

        quant_file = tflite_path.parent / "output_quant.txt"
        quant_file.write_text(f"{info['output_scale']},{info['output_zero_point']}")
        print(f"\n✅ output_quant.txt guardado en: {quant_file}")

Parámetros del modelo:
  Input  shape     : [1, 96, 96, 3]
  Input  scale/zp  : 0.003922 / -128
  Output shape     : [1, 84, 189]
  Output scale/zp  : 0.006987 / -128

✅ output_quant.txt guardado en: yolov8n_saved_model/output_quant.txt


# Generar model_data.cc para Arduino

In [12]:
output_cc = Path("arduino/tier1_sketch/model_data.cc")
output_cc.parent.mkdir(parents=True, exist_ok=True)

print(f"Generando {output_cc}...")
print("(30-60 segundos para un modelo de ~3-4 MB)\n")

with open(tflite_path, "rb") as f:
    model_bytes = f.read()

# Formatear como array C
hex_vals = ", ".join(f"0x{b:02x}" for b in model_bytes)

cc_content = f"""// Auto-generado por el notebook — no editar manualmente
// Fuente: {tflite_path.name}  ({len(model_bytes):,} bytes)
#include "model_data.h"

const unsigned char g_model[] = {{{hex_vals}}};
const unsigned int  g_model_len = {len(model_bytes)};
"""

output_cc.write_text(cc_content)

size_mb = output_cc.stat().st_size / (1024 * 1024)
print(f"✅ Generado: {output_cc}")
print(f"   Tamaño del .cc: {size_mb:.1f} MB")
print()
print("Siguiente paso:")
print("  1. Abrir Arduino IDE → tier1_sketch/tier1_sketch.ino")
print("  2. Tools → Board → XIAO_ESP32S3")
print("  3. Tools → PSRAM → OPI PSRAM  ← OBLIGATORIO")
print("  4. Upload (compilación ~8-10 min la primera vez)")

Generando arduino/tier1_sketch/model_data.cc...
(30-60 segundos para un modelo de ~3-4 MB)

✅ Generado: arduino/tier1_sketch/model_data.cc
   Tamaño del .cc: 19.4 MB

Siguiente paso:
  1. Abrir Arduino IDE → tier1_sketch/tier1_sketch.ino
  2. Tools → Board → XIAO_ESP32S3
  3. Tools → PSRAM → OPI PSRAM  ← OBLIGATORIO
  4. Upload (compilación ~8-10 min la primera vez)


In [1]:
from pathlib import Path

archivos = [
    Path("arduino/tier1_sketch/tier1_sketch.ino"),
    Path("arduino/tier1_sketch/model_data.h"),
    Path("arduino/tier1_sketch/model_data.cc"),
]

for f in archivos:
    if f.exists():
        size = f.stat().st_size / 1024
        print(f"✅ {f.name:30s} {size:>8.1f} KB")
    else:
        print(f"❌ {f.name:30s} NO ENCONTRADO — {f}")

✅ tier1_sketch.ino                    4.8 KB
✅ model_data.h                        0.9 KB
✅ model_data.cc                   19823.1 KB


In [2]:
partitions_content = """# 8MB Flash — 4MB APP + 4MB SPIFFS
# Name,   Type, SubType, Offset,   Size
nvs,      data, nvs,     0x9000,   0x5000,
otadata,  data, ota,     0xe000,   0x2000,
app0,     app,  ota_0,   0x10000,  0x3F0000,
spiffs,   data, spiffs,  0x400000, 0x400000,
"""

partitions_file = Path("arduino/tier1_sketch/partitions.csv")
partitions_file.write_text(partitions_content)
print(f"✅ Creado: {partitions_file}")
print(f"   App partition: 3.94 MB  (sketch necesita 3.78 MB — margen OK)")

✅ Creado: arduino/tier1_sketch/partitions.csv
   App partition: 3.94 MB  (sketch necesita 3.78 MB — margen OK)


In [3]:
# pyserial y opencv no se instalaron antes — agregarlos ahora
result = subprocess.run(
    [str(pip312), "install", "--quiet", "pyserial", "opencv-python"],
    capture_output=True, text=True, env=clean_env
)
print("✅ pyserial + opencv instalados" if result.returncode == 0 else result.stderr)

NameError: name 'subprocess' is not defined

# Correr pipeline

In [14]:
import subprocess 
port = "/dev/cu.usbmodem1101"  # tu puerto

result = subprocess.run([
    str(python312_venv),
    "tier1_pipeline.py",
    "--port", "/dev/cu.usbmodem1101",
    "--all-gammas",
    "--n-images", "5"       # 5 imágenes × 5 gammas = 25 inferences × 7.7s ≈ 3 min
], capture_output=False, text=True, env=clean_env, cwd=Path(".").resolve())

COCO128: 128 imágenes en /Users/zafika/Documents/Universidad/2026-1/Vision Por Computador/datasets/coco128/images/train2017
⚠️   Modo debug: usando solo 5 imagen(es)
Conectando a /dev/cu.usbmodem1101 @ 921600 baud…


Traceback (most recent call last):
  File "/Users/zafika/Documents/Universidad/2026-1/Vision Por Computador/Tarea-2-Vision-Computador/tier1_pipeline.py", line 511, in <module>
    main()
  File "/Users/zafika/Documents/Universidad/2026-1/Vision Por Computador/Tarea-2-Vision-Computador/tier1_pipeline.py", line 460, in main
    ser = connect_to_esp32(args.port)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/zafika/Documents/Universidad/2026-1/Vision Por Computador/Tarea-2-Vision-Computador/tier1_pipeline.py", line 68, in connect_to_esp32
    if ser.in_waiting:
       ^^^^^^^^^^^^^^
  File "/Users/zafika/Documents/Universidad/2026-1/Vision Por Computador/Tarea-2-Vision-Computador/venv_export312/lib/python3.12/site-packages/serial/serialposix.py", line 549, in in_waiting
    s = fcntl.ioctl(self.fd, TIOCINQ, TIOCM_zero_str)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 6] Device not configured
